# Swin Object Detection — DIMER TASK-INFERENCE tutorial

**Profile:** `TASK-INFERENCE`  
**Notebook spec:** DIMER Notebook Specification 1.0  
**Capability:** pretrained object detection with the repository's public `DimerSwinDetector` API.

This tutorial runs the pinned **Swin-T + Mask R-CNN** detector through the repository-owned task-inference runtime. **No fine-tuning occurs.** The model maps an RGB image to class-labelled axis-aligned boxes and uncalibrated class confidence scores. The v1 DIMER contract deliberately excludes the checkpoint's instance-mask output.

By the end you will be able to bootstrap the exact runtime, verify the checkpoint before loading it, validate images, run detection, evaluate a labelled tutorial sample with COCO AP/AP50/AP75, optionally score your own image, and export machine-readable detections, metrics, and provenance.

The default sample is the public COCO8 mini-dataset distributed by Ultralytics (eight COCO 2017 images; this notebook evaluates its validation subset). Its metrics are **tutorial sanity evidence only**, not reproduction of the upstream COCO benchmark and not production-fitness evidence.


## Prerequisites and trust boundary

- **Supported runtime:** CPython 3.10 Jupyter on Linux; CPU is the default path. GPU is not required.
- **Network:** GitHub, PyPI/OpenMMLab package indexes, the OpenMMLab checkpoint host, and the pinned COCO8 release asset.
- **Model serialization:** the upstream `.pth` file is code-capable PyTorch serialization. The repository verifies its exact byte size and SHA-256 **before** MMDetection loads it. A matching digest establishes byte identity, not sender authenticity; use only the pinned trusted upstream source.
- **BYOD:** optional and disabled by default. Do not upload confidential or restricted imagery to a notebook environment you are not authorized to use. Input images remain in the notebook runtime and are not sent to an inference service.

Current OpenMMLab wheels for this frozen stack require Python 3.10 and NumPy 1.x. The notebook fails clearly rather than silently changing the model stack.


## 1. Bootstrap the immutable repository runtime

This cell checks out the exact repository revision that implements `DimerSwinDetector`, installs one pinned CPU dependency graph, installs the repository package without resolving another graph, and prints the effective versions. **Look for:** Python 3.10, torch 2.1.2, MMDetection 3.3.0, MMCV 2.1.0, MMEngine 0.10.7, NumPy 1.26.4.


In [ ]:
from __future__ import annotations
import csv, hashlib, json, os, shutil, stat, subprocess, sys, urllib.request, zipfile
from pathlib import Path, PurePosixPath

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 is required by the qualified OpenMMLab runtime; current interpreter is {sys.version.split()[0]}. Use a Python 3.10 Jupyter kernel.')

REPOSITORY = 'https://github.com/kurtvalcorza/swin-detection-pipeline.git'
REPOSITORY_REF = 'b2c220e3dd4678e8c1a1f9982251cf31a796fc45'
WORK = Path(os.environ.get('DIMER_TUTORIAL_WORKSPACE', '/tmp/dimer-swin-detection-tutorial')).resolve()
REPO = WORK / 'repo'
OUTPUTS = WORK / 'outputs'
WORK.mkdir(parents=True, exist_ok=True); OUTPUTS.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    cmd = [str(x) for x in cmd]; print('+', ' '.join(cmd)); subprocess.run(cmd, cwd=cwd, check=True)

if not REPO.exists():
    run(['git','clone','--filter=blob:none',REPOSITORY,REPO])
run(['git','checkout','--detach',REPOSITORY_REF], cwd=REPO)
head = subprocess.run(['git','rev-parse','HEAD'], cwd=REPO, check=True, capture_output=True, text=True).stdout.strip()
assert head == REPOSITORY_REF

run([sys.executable,'-m','pip','install','--disable-pip-version-check','--index-url','https://download.pytorch.org/whl/cpu','torch==2.1.2','torchvision==0.16.2'])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','openmim==0.3.9'])
run(['mim','install','mmengine==0.10.7'])
run(['mim','install','mmcv==2.1.0'])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','mmdet==3.3.0','pycocotools==2.0.11'])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','--force-reinstall','numpy==1.26.4','opencv-python==4.10.0.84'])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','--no-deps',REPO])
run([sys.executable,'-m','pip','check'])

import importlib.metadata as md, numpy as np, torch
runtime = {'python':sys.version.split()[0],'torch':torch.__version__,'numpy':np.__version__,'mmdet':md.version('mmdet'),'mmcv':md.version('mmcv'),'mmengine':md.version('mmengine'),'repositoryRevision':REPOSITORY_REF}
print(json.dumps(runtime, indent=2))


## 2. Acquire and validate the labelled tutorial sample

The sample archive is fetched from an immutable GitHub release URL. Extraction is path-safe and size-capped. We use the COCO8 **validation** images and their YOLO-format labels only; the notebook does not resplit data. Reported values are sample metrics, not COCO benchmark metrics.


In [ ]:
COCO8_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip'
archive = WORK / 'coco8.zip'; sample_root = WORK / 'sample'
if not archive.exists(): urllib.request.urlretrieve(COCO8_URL, archive)
sample_sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()

def safe_extract(zpath: Path, root: Path, max_expanded=25*1024*1024):
    if root.exists(): shutil.rmtree(root)
    root.mkdir(parents=True)
    with zipfile.ZipFile(zpath) as zf:
        total = 0
        for info in zf.infolist():
            p = PurePosixPath(info.filename)
            if p.is_absolute() or '..' in p.parts or '\\' in info.filename: raise ValueError(f'unsafe archive member: {info.filename}')
            if stat.S_ISLNK(info.external_attr >> 16): raise ValueError(f'symlink refused: {info.filename}')
            total += info.file_size
            if total > max_expanded: raise ValueError('archive exceeds expanded-size ceiling')
        zf.extractall(root)
safe_extract(archive, sample_root)
dataset = sample_root / 'coco8'
images = sorted((dataset/'images'/'val').glob('*'))
labels_dir = dataset/'labels'/'val'
if not images: raise RuntimeError('COCO8 validation images were not found after extraction.')
from dimer_swin_detection.runtime import validate_image
sample_table = [validate_image(p) for p in images]
print(json.dumps({'sampleType':'public COCO8/COCO 2017 subset','archiveSha256':sample_sha256,'validationImages':sample_table}, indent=2))


## 3. Resolve the pinned model and run the real repository API

`DimerSwinDetector` downloads the exact OpenMMLab checkpoint, checks its expected 191,461,353-byte size and SHA-256 `9d6b7cfa…a193a291`, then passes it to MMDetection. Predictions below use `score_threshold=0.0` so evaluation sees the detector output rather than a display threshold. Scores are **uncalibrated**; a score of 0.9 is not a 90% probability that a box is correct.


In [ ]:
from dimer_swin_detection import DimerSwinDetector, MODEL_SPEC
detector = DimerSwinDetector(cache_dir=WORK/'models', device='cpu')
prediction_rows = [d.to_dict() for d in detector.predict_many(images, score_threshold=0.0, max_detections=300)]
(OUTPUTS/'detections.json').write_text(json.dumps(prediction_rows, indent=2)+'\n')
with (OUTPUTS/'detections.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['image_id','class_id','class_name','score','x1','y1','x2','y2']); writer.writeheader()
    for row in prediction_rows:
        x1,y1,x2,y2=row['bbox_xyxy']; writer.writerow({'image_id':row['image_id'],'class_id':row['class_id'],'class_name':row['class_name'],'score':row['score'],'x1':x1,'y1':y1,'x2':x2,'y2':y2})
print(json.dumps({'model':MODEL_SPEC['runtime_id'],'detections':len(prediction_rows),'top5':prediction_rows[:5]}, indent=2))


## 4. Evaluate COCO AP on the frozen tutorial subset

Ground-truth YOLO boxes are converted to COCO coordinates without changing image membership. We report COCO AP@[0.50:0.95], AP50, and AP75 because they are the repository's specified detection metrics. The empty-detector baseline has AP=0 by construction. With only four validation images, these values have high sampling variance and are **not** comparable to the upstream full-COCO result.


In [ ]:
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
COCO_CATEGORY_IDS=[1,2,3,4,5,6,7,8,9,10,11,13,14,15,16,17,18,19,20,21,22,23,24,25,27,28,31,32,33,34,35,36,37,38,39,40,41,42,43,44,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,67,70,72,73,74,75,76,77,78,79,80,81,82,84,85,86,87,88,89,90]
assert len(COCO_CATEGORY_IDS)==len(detector.classes)==80
coco_images=[]; annotations=[]; image_id_by_name={}; ann_id=1
for image_id, path in enumerate(images, start=1):
    with Image.open(path) as im: width,height=im.size
    image_id_by_name[path.name]=image_id; coco_images.append({'id':image_id,'file_name':path.name,'width':width,'height':height})
    label_file=labels_dir/(path.stem+'.txt')
    for line in label_file.read_text().splitlines():
        cls,xc,yc,bw,bh=map(float,line.split()); cls=int(cls); w=bw*width; h=bh*height; x=xc*width-w/2; y=yc*height-h/2
        annotations.append({'id':ann_id,'image_id':image_id,'category_id':COCO_CATEGORY_IDS[cls],'bbox':[x,y,w,h],'area':w*h,'iscrowd':0}); ann_id+=1
gt=COCO(); gt.dataset={'info':{'description':'COCO8 validation tutorial subset'},'images':coco_images,'annotations':annotations,'categories':[{'id':cid,'name':detector.classes[i]} for i,cid in enumerate(COCO_CATEGORY_IDS)]}; gt.createIndex()
results=[]
for row in prediction_rows:
    x1,y1,x2,y2=row['bbox_xyxy']; results.append({'image_id':image_id_by_name[row['image_id']],'category_id':COCO_CATEGORY_IDS[row['class_id']],'bbox':[x1,y1,x2-x1,y2-y1],'score':row['score']})
dt=gt.loadRes(results); evaluator=COCOeval(gt,dt,'bbox'); evaluator.params.imgIds=sorted(image_id_by_name.values()); evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
metrics={'estimation':'single frozen COCO8 validation subset','num_images':len(images),'num_ground_truth_boxes':len(annotations),'coco_ap_50_95':float(evaluator.stats[0]),'ap50':float(evaluator.stats[1]),'ap75':float(evaluator.stats[2]),'empty_detector_baseline_ap':0.0,'upstream_full_coco_box_ap_not_measured_here':MODEL_SPEC['upstream_reported_box_ap']}
(OUTPUTS/'metrics.json').write_text(json.dumps(metrics,indent=2)+'\n'); print(json.dumps(metrics,indent=2))


## 5. Optional BYOD new-image inference

This branch is disabled by default so the sample path never blocks on an upload dialog. Set `USE_BYOD=True` and provide `BYOD_PATH`; in Colab, leaving the path empty opens the upload picker. The repository validates readability and a 64-megapixel ceiling before model execution. BYOD has no metric unless you independently provide ground truth; this section is inference only.


In [ ]:
USE_BYOD=False  # @param {type:'boolean'}
BYOD_PATH=''   # @param {type:'string'}
DISPLAY_SCORE_THRESHOLD=0.5  # @param {type:'number'}
byod_predictions=[]
if USE_BYOD:
    p=Path(BYOD_PATH).expanduser() if BYOD_PATH else None
    if p is None:
        try:
            from google.colab import files
            uploaded=files.upload(); p=Path(next(iter(uploaded))); p.write_bytes(uploaded[p.name])
        except Exception as exc:
            raise RuntimeError('Set BYOD_PATH to a local image, or use Colab upload.') from exc
    from dimer_swin_detection.runtime import validate_image
    print(validate_image(p)); byod_predictions=[d.to_dict() for d in detector.predict(p,score_threshold=DISPLAY_SCORE_THRESHOLD)]
    (OUTPUTS/'byod-detections.json').write_text(json.dumps(byod_predictions,indent=2)+'\n'); print(json.dumps(byod_predictions[:10],indent=2))
else:
    print('BYOD skipped (default).')


## 6. Export provenance and verify outputs

The provenance record binds the effective repository runtime, OpenMMLab versions, model identifier, checkpoint digest, class ordering, score semantics, sample identity, and tutorial metrics. It contains no credentials. Machine-readable detections and metrics are written separately so downstream systems do not need notebook state.


In [ ]:
provenance=detector.provenance(); provenance['tutorial']={'notebookProfile':'TASK-INFERENCE','notebookSpec':'1.0','repositoryRevision':REPOSITORY_REF,'sampleUrl':COCO8_URL,'sampleArchiveSha256':sample_sha256,'metrics':metrics}
(OUTPUTS/'provenance.json').write_text(json.dumps(provenance,indent=2)+'\n')
required=['detections.json','detections.csv','metrics.json','provenance.json']
missing=[name for name in required if not (OUTPUTS/name).is_file()]
if missing: raise RuntimeError(f'Missing expected outputs: {missing}')
print(json.dumps({'outputs':[str(OUTPUTS/n) for n in required],'checkpointVerified':provenance['effective']['checkpoint_sha256']==MODEL_SPEC['checkpoint_sha256']},indent=2))


## Interpretation, limits, and next experiments

A successful run proves that this repository revision can acquire the pinned pretrained Swin-T + Mask R-CNN bytes, verify them, reconstruct the official MMDetection inference path, validate and score new RGB images, compute task-appropriate metrics on a labelled sample, and export portable results/provenance. It does **not** prove ImageNet/COCO benchmark reproduction, score calibration, robustness to your domain, fairness, safety, production fitness, or support for training/fine-tuning.

The displayed sample AP is intentionally a small-sample sanity check. For a deployment decision, use a representative labelled holdout from the target domain, inspect per-class/per-size failure modes, calibrate score/NMS operating points, and evaluate distribution shifts. A useful next experiment is to run BYOD on several images from one target domain and compare qualitative misses before investing in labelled evaluation data.
